# Inicio rápido: datos municipales y mapa interactivo

Este notebook busca una variable, descarga datos desde SINIM y construye un mapa comunal interactivo con [Kepler.gl](https://kepler.gl/). El ejemplo utiliza los ingresos por patentes municipales (`4173`) para el último año disponible.

## 1. Instalación

`keplergl` se instala solo para este ejemplo visual; no es necesario para el uso habitual de `mcp-sinim`. Este notebook requiere **Python 3.10, 3.11 o 3.12**, porque las dependencias actuales de Kepler.gl todavía no son compatibles con Python 3.13. Reinicia el kernel si Jupyter lo solicita.

In [ ]:
%pip install --quiet --upgrade mcp-sinim keplergl

## 2. Buscar y descargar datos SINIM

In [ ]:
from mcp_sinim import SINIMClient

client = SINIMClient(corrmon=True)
client.search("patentes municipales", limit=5)[["code", "name", "unit", "score"]]

In [ ]:
codigo = "4173"
ultimo_anio = client.years()[-1]

datos = client.get(codigo, years=[ultimo_anio])
datos = datos.dropna(subset=["value"]).copy()
print(f"{len(datos)} municipios con datos para {ultimo_anio}")
datos.head()

## 3. Unir los datos con polígonos comunales

Las geometrías provienen del repositorio [Chile-GeoJSON](https://github.com/fcortes/Chile-GeoJSON), elaborado a partir de mapas vectoriales de la Biblioteca del Congreso Nacional de Chile. La unión se realiza mediante el código CUT comunal.

In [ ]:
import json
from urllib.request import urlopen

GEOJSON_URL = (
    "https://raw.githubusercontent.com/fcortes/"
    "Chile-GeoJSON/master/comunas.geojson"
)

with urlopen(GEOJSON_URL) as respuesta:
    comunas = json.load(respuesta)

por_comuna = datos.assign(
    cod_municipio=datos["cod_municipio"].astype(str).str.zfill(5)
).set_index("cod_municipio")

features = []
for feature in comunas["features"]:
    codigo_cut = str(feature["properties"]["cod_comuna"]).zfill(5)
    if codigo_cut not in por_comuna.index:
        continue
    fila = por_comuna.loc[codigo_cut]
    feature["properties"].update(
        {
            "cod_municipio": codigo_cut,
            "municipio": fila["nombre_municipio"],
            "indicador": fila["name"],
            "valor": float(fila["value"]),
            "unidad": fila["unit"],
            "anio": int(fila["year"]),
        }
    )
    features.append(feature)

mapa_geojson = {"type": "FeatureCollection", "features": features}
print(f"{len(features)} polígonos comunales listos para visualizar")

## 4. Crear el mapa interactivo

En el panel lateral de Kepler.gl puedes colorear los polígonos usando `valor`, filtrar por `Region` y consultar cada comuna al pasar el cursor.

In [ ]:
from keplergl import KeplerGl

mapa = KeplerGl(
    height=700,
    data={f"Ingresos por patentes {ultimo_anio}": mapa_geojson},
    show_docs=False,
)
mapa

## 5. Exportar el resultado (opcional)

El archivo HTML conserva la interacción y puede abrirse en cualquier navegador. No se incluye en el repositorio porque contiene todos los datos y geometrías del mapa.

In [ ]:
mapa.save_to_html(
    file_name=f"mapa_patentes_sinim_{ultimo_anio}.html",
    read_only=True,
)
client.close()